# Notebook 1 – Panel Data Cleaning

This notebook loads the raw sensory panel Excel file, applies all structural and data-quality cleaning steps, and saves a single **taster-level** cleaned dataset for use in Notebook 2.

The following steps are happening in **Notebook 2**,including the descriptive analysis:
- Removing Status 3 rows
- Filtering to the final set of target sections
- Aggregation 
- Modelling

| Output file | Description |
|---|---|
| `panel_cleaned.csv` | Fully cleaned taster-level panel data — all sections and statuses retained |

### Cleaning pipeline overview
1. Load raw Excel file and fix raw structure (header row, `x` columns)
2. Combine all trial sheets into one DataFrame
3. Cast data types
4. Fix taster name typo (`PB` → `BP`)
5. Correct Header and Section assignments and standardise names
6. Set Criteria = 'Comment' for pure comment sections
7. Validate and repair Grade values
8. Fix Group value `x` → `Savoury`
9. Detect and resolve duplicate grades per taster/section
10. Normalise Group names 
11. Fix incorrect `Header_ID` and `Section_ID` values
12. Validate taster completeness per trial-section
13. Save `panel_cleaned.csv`

---
## 0 · Configuration
All parameters are gathered here. You only need to edit this cell when you add new trials or change the set of target sections.

In [23]:
# ============================================================
# PATH TO THE RAW PANEL EXCEL FILE
# ============================================================
PANEL_FILE_PATH = r"C:/Users/miria/Downloads/Sensory 20251208-09 NFI Cheddar Tasting Analysis all data.xlsx"

# ============================================================
# SHEETS TO EXCLUDE (not trial data)
# ============================================================
EXCLUDED_SHEETS = ["Explainer", "Reference page"]

# ============================================================
# EXPECTED TASTERS (used for completeness check)
# ============================================================
EXPECTED_TASTERS = {"CT", "PM", "DB", "BP", "LH"}

# ============================================================
# OUTPUT PATH
# The cleaned taster-level data is saved here.
# Notebook 2 loads this file to run descriptive analysis,
# apply section filters, aggregate, and model.
# ============================================================
OUT_PANEL_CLEANED = "panel_cleaned.csv"

print("Configuration loaded successfully.")
print(f"  Output file: {OUT_PANEL_CLEANED}")

Configuration loaded successfully.
  Output file: panel_cleaned.csv


---
## 1 · Imports

In [24]:
import re
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

# Display settings: show all columns / rows in DataFrames
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows",    None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width",        None)

# ──────────────────────────────────────────────────────────────
# Helper: log a cleaning step with before/after row counts
# ──────────────────────────────────────────────────────────────
def log_step(df_before, df_after, step_name: str) -> None:
    removed = len(df_before) - len(df_after)
    print(f"[{step_name}]  rows before: {len(df_before):,}  |  "
          f"removed: {removed:,}  |  rows after: {len(df_after):,}")

print("Imports done.")

Imports done.


---
## 2 · Load Raw Panel Data

In [25]:
# Open the workbook and list all sheet names
xl = pd.ExcelFile(PANEL_FILE_PATH)
print("All sheets found in the file:")
print(xl.sheet_names)

# Filter out non-data sheets
trial_sheets = [s for s in xl.sheet_names if s not in EXCLUDED_SHEETS]
print(f"\nSheets used as trial data ({len(trial_sheets)}): {trial_sheets}")

All sheets found in the file:
['Explainer', '110009', '110001', '120003', '110024', '110002', '110022', '120004', '110021', '110019', '240988', '110020', '110023', 'Reference page']

Sheets used as trial data (12): ['110009', '110001', '120003', '110024', '110002', '110022', '120004', '110021', '110019', '240988', '110020', '110023']


In [26]:
# ──────────────────────────────────────────────────────────────
# The raw sheets have the following layout:
#   row 0      : some metadata / description row  (skip)
#   row 1      : column headers  
#   rows 2-3   : more metadata rows  (skip)
#   row 4+     : actual panel data
#
# Each sheet name is the trial number (e.g. '110009').
# Read every sheet into a dict: {trial_id: DataFrame}
# ──────────────────────────────────────────────────────────────

panel_data = pd.read_excel(xl, sheet_name=trial_sheets)

for trial_id, df_raw in panel_data.items():
    # Set the header from row 1 (index 1)
    df_clean = df_raw.iloc[4:].copy()          # data starts at row 4
    df_clean.columns = df_raw.iloc[1]          # row 1 has the real column names
    df_clean.columns.name = None               # remove the multi-index name
    df_clean = df_clean.reset_index(drop=True) # reset row index

    # Remove the helper column 'x' that the tasting software exports
    if "x" in df_clean.columns:
        df_clean = df_clean.drop(columns=["x"])

    panel_data[trial_id] = df_clean

# Shape check per trial
print("Shape per trial (rows × columns):")
for t, df in panel_data.items():
    print(f"  {t}: {df.shape}")

Shape per trial (rows × columns):
  110009: (308, 12)
  110001: (331, 12)
  120003: (345, 12)
  110024: (316, 12)
  110002: (286, 12)
  110022: (257, 12)
  120004: (277, 12)
  110021: (276, 12)
  110019: (312, 12)
  240988: (326, 12)
  110020: (282, 12)
  110023: (338, 12)


---
## 3 · Combine All Trials into One DataFrame

In [27]:
panel_all = pd.concat(panel_data.values(), ignore_index=True)

# Rename columns 
panel_all = panel_all.rename(columns={
    "Sample"          : "Trial",
    "H#"              : "Header_ID",
    "S#"              : "Section_ID",
    "Grade description": "Grade_Description",
})

print(f"Combined DataFrame: {panel_all.shape[0]:,} rows × {panel_all.shape[1]} columns")
print("\nColumns:", list(panel_all.columns))
panel_all.head(3)

Combined DataFrame: 3,654 rows × 12 columns

Columns: ['Header_ID', 'Header', 'Section_ID', 'Section', 'Criteria', 'Trial', 'Taster', 'Status', 'Group', 'Comment', 'Grade', 'Grade_Description']


,Header_ID,Header,Section_ID,Section,Criteria,Trial,Taster,Status,Group,Comment,Grade,Grade_Description
0,1,Appearance,1,Visual comments,Comment,110009,BP,1,NaN,"Slight moisture, some cracks, slightly crumbly edges, breaks smoothly, no crumbs",NaN,"[comment, not graded]"
1,1,Appearance,2,Paste structure,Obs,110009,BP,1,Crumbly,Crumbly edges,2,Mild
2,1,Appearance,2,Paste structure,Obs,110009,BP,1,Smooth,Smooth,3,Moderate


---
## 4 · Data Type Casting

In [28]:
# ── String columns
# Header_ID and Section_ID are identifiers — stored as string
# for consistency with Trial and to avoid numeric operations.
string_cols = ["Trial", "Header", "Section", "Criteria",
               "Taster", "Group", "Comment", "Grade_Description",
               "Header_ID", "Section_ID"]
panel_all[string_cols] = panel_all[string_cols].astype("string")

# ── Numeric grade: kept as float (grades of 0, 0.5 and NaN can occur)
panel_all["Grade"] = pd.to_numeric(panel_all["Grade"], errors="coerce")

# ── Status: kept as integer (1, 2, 3 — used for filtering)
panel_all["Status"] = pd.to_numeric(panel_all["Status"], errors="coerce").astype("Int64")

print("Data types after casting:")
display(panel_all.dtypes.to_frame(name="dtype"))

Data types after casting:


,dtype
Header_ID,string[python]
Header,string[python]
Section_ID,string[python]
Section,string[python]
Criteria,string[python]
Trial,string[python]
Taster,string[python]
Status,Int64
Group,string[python]
Comment,string[python]


---
## 5 · Fix Taster Name Typo (`PB` → `BP`)

In [29]:
# The file says 'PB' instead of 'BP' for one taster.
# This step corrects the typo so every taster has a unique, consistent ID.

print("Taster counts BEFORE fix:")
display(panel_all["Taster"].value_counts())

panel_all["Taster"] = panel_all["Taster"].replace("PB", "BP")

print("\nTaster counts AFTER fix:")
display(panel_all["Taster"].value_counts())

assert panel_all["Taster"].nunique() == len(EXPECTED_TASTERS), \
    f"Expected {len(EXPECTED_TASTERS)} tasters, found {panel_all['Taster'].nunique()}"
print("\n Taster count is correct.")

Taster counts BEFORE fix:


Taster
CT    772
PM    742
DB    727
BP    708
LH    688
PB     17
Name: count, dtype: Int64


Taster counts AFTER fix:


Taster
CT    772
PM    742
DB    727
BP    725
LH    688
Name: count, dtype: Int64


 Taster count is correct.


---
## 6 · Fix Header & Section Assignments, Standardise Names

Some sections were placed under the wrong header in the raw data (e.g. 'Aroma Intensity' listed under 'Appearance' instead of 'Nose'). Additionally, sections with the same name appear under different headers (e.g. 'Aromas/Flavours, Dairy' exists for both Nose and Palate), so they need unique names.

In [30]:
# ── Step 6a: Move sections to their correct Header ──────────────────────────

# 'Palate intensity' was incorrectly placed under 'Tastes' → move to 'Palate'
panel_all.loc[
    (panel_all["Header"] == "Tastes") & (panel_all["Section"] == "Palate intensity"),
    "Header"
] = "Palate"

# Aroma-related sections were incorrectly placed under 'Appearance' → move to 'Nose'
for wrong_section in ["Aroma Intensity", "Aromas/Flavours, Dairy", "Aromas/Flavours, Other"]:
    panel_all.loc[
        (panel_all["Header"] == "Appearance") & (panel_all["Section"] == wrong_section),
        "Header"
    ] = "Nose"

# ── Step 6b: Give ambiguous sections unique names ───────────────────────────
# Sections that share a generic name across multiple headers get a prefixed name
# (e.g. 'Aromas/Flavours, Dairy' becomes 'Nose_Aromas_Dairy' or 'Palate_Aromas_Dairy')

SECTION_RENAME_MAP = {
    ("Nose",          "Aromas/Flavours, Dairy")       : "Nose_Aromas_Dairy",
    ("Nose",          "Aromas/Flavours, Other")        : "Nose_Aromas_Other",
    ("Nose",          "Aroma Intensity")                : "Nose_Aroma_Intensity",
    ("Palate",        "Aromas/Flavours, Dairy")         : "Palate_Aromas_Dairy",
    ("Palate",        "Aromas/Flavours, Other")         : "Palate_Aromas_Other",
    ("AQ",            "Assessment of Quality/total")   : "AQ_Total",
    ("Client extras", "Does it taste processed")        : "Processed_Taste",
    ("Context",       "Evidence of context")           : "Context_Evidence",
    ("Context",       "Quality in context")            : "Context_Quality",
    ("Context",       "Evidence of quality in context"): "Context_Evidence_Quality",
}

for (header, section), new_name in SECTION_RENAME_MAP.items():
    mask = (panel_all["Header"] == header) & (panel_all["Section"] == section)
    panel_all.loc[mask, "Section"] = new_name

# ── Step 6c: Standardise all Header and Section names ─────────────────────
# Rule: spaces/slashes/commas → underscores, first letter of each word capitalised,
# 'of' stays lowercase when not the first word, sequences of underscores collapsed.

def standardise_name(text):
    """Convert a raw panel label to a clean Title_Case_With_Underscores string."""
    if not isinstance(text, str):
        return text
    # Replace separators (space, slash, comma, plus, ?) with underscores
    text = re.sub(r'[ /,+?]+', '_', text)
    # Collapse multiple underscores and strip leading/trailing ones
    text = re.sub(r'_+', '_', text).strip('_')
    # Capitalise each word, keeping 'of' lowercase unless it's the first word
    words = text.split('_')
    formatted = []
    for i, word in enumerate(words):
        if word.lower() == "of" and i != 0:
            formatted.append("of")
        elif len(word) > 0:
            formatted.append(word[0].upper() + word[1:])   # keeps AQ, GCMS-style acronyms intact
        else:
            formatted.append(word)
    return "_".join(formatted)

panel_all["Header"]  = panel_all["Header"].apply(standardise_name)
panel_all["Section"] = panel_all["Section"].apply(standardise_name)

print("Section–Header mapping after renaming:")
display(pd.crosstab(panel_all["Header"], panel_all["Section"]))

Section–Header mapping after renaming:


Section,AQ_Total,Acid_Sour,Aftertastes,Balance,Bitter,Complexity,Condition,Context,Context_Evidence,Context_Evidence_Quality,Context_Quality,Degree_of_Artificiality,Development,Interest,Length,Mouthfeel_Chewed,Mouthfeel_Initial,Nose_Aroma_Intensity,Nose_Aromas_Dairy,Nose_Aromas_Other,Palate_Aromas_Dairy,Palate_Aromas_Other,Palate_Intensity,Paste_Colour,Paste_Structure,Processed_Taste,Salt,Satisfaction,Savoury_Umami,Sweet,Trigeminal_Intensity,Trigeminal_Notes,Visual_Comments
Header,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
AQ,60,0,0,60,0,60,60,0,0,0,0,0,0,0,60,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
Appearance,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,104,312,0,0,0,0,0,0,0,64
Client_Extras,0,0,60,0,0,0,0,0,0,0,0,60,0,0,0,0,0,0,0,0,0,0,0,0,0,60,0,0,0,0,0,0,0
Conclusion,0,0,0,0,0,0,0,0,0,0,0,0,60,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
Context,0,0,0,0,0,0,0,60,60,60,60,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
Nose,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,60,286,315,0,0,0,0,0,0,0,0,0,0,0,0,0
Palate,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,155,127,0,0,0,314,432,59,0,0,0,0,0,0,0,0,165,0
Satisfaction_Interest,0,0,0,0,0,0,0,0,0,0,0,0,0,60,0,0,0,0,0,0,0,0,0,0,0,0,0,60,0,0,0,0,0
Tastes,0,60,0,0,61,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,60,0,60,60,60,0,0


---
## 7 · Set Criteria = 'Comment' for Pure Comment Sections

Some sections (e.g. `Visual_Comments`, `Aftertastes`) should always have `Criteria = 'Comment'` because they never carry a numeric grade. Ensuring this is consistent avoids them being treated as scored sections in later steps.

In [31]:
COMMENT_ONLY_SECTIONS = [
    "Visual_Comments",
    "Aftertastes",
    "Context",
    "Context_Evidence",
    "Context_Evidence_Quality",
]

mask_comment = panel_all["Section"].isin(COMMENT_ONLY_SECTIONS)
rows_updated  = mask_comment.sum()
panel_all.loc[mask_comment, "Criteria"] = "Comment"

print(f"Updated Criteria to 'Comment' for {rows_updated:,} rows in {COMMENT_ONLY_SECTIONS}.")

Updated Criteria to 'Comment' for 304 rows in ['Visual_Comments', 'Aftertastes', 'Context', 'Context_Evidence', 'Context_Evidence_Quality'].


---
## 8 · Grade Validation & Repair

Three specific issues are addressed:
- **Comment 'x'**: these rows carry no data and are dropped.
- **Comment '0'**: the taster wrote '0' in the Comment field instead of entering grade 0; the value is moved to Grade.
- **'Absent' inconsistency**: rows flagged as 'Absent' but with a grade > 0.5 are set to grade 0.

Also verify that `AQ_Total` equals `Balance + Complexity + Condition + Length` for every taster.

In [32]:
n_rows_start = len(panel_all)

# ── AQ Total integrity check ───────────
AQ_COMPONENTS = ["Balance", "Complexity", "Condition", "Length", "AQ_Total"]
df_aq = panel_all[panel_all["Section"].isin(AQ_COMPONENTS)].pivot_table(
    index=["Trial", "Taster"], columns="Section", values="Grade"
).reset_index()

if all(c in df_aq.columns for c in AQ_COMPONENTS):
    df_aq["Calculated_Sum"] = df_aq[["Balance", "Complexity", "Condition", "Length"]].sum(axis=1)
    df_aq["Difference"]     = df_aq["AQ_Total"] - df_aq["Calculated_Sum"]
    aq_errors = df_aq[df_aq["Difference"].abs() > 0.01]
    if aq_errors.empty:
        print(" AQ_Total is consistent with the sum of its components for all tasters.")
    else:
        print(f"  {len(aq_errors)} rows where AQ_Total ≠ component sum:")
        display(aq_errors[["Trial", "Taster", "AQ_Total", "Calculated_Sum", "Difference"]])

# ── Step 1: Remove rows where Comment == 'x' ─────────────
mask_x = panel_all["Comment"].astype(str).str.strip() == "x"
n_x    = mask_x.sum()
panel_all = panel_all.loc[~mask_x].copy()
print(f"\nStep 1 – Removed {n_x:,} rows where Comment == 'x'.  Rows remaining: {len(panel_all):,}")

# ── Step 2: Convert Comment == '0' → Grade = 0, but only when Grade is missing
# Rationale: if a taster already has a numeric grade, the '0' in Comment is
# likely a remark, not a missing grade value. Only fill in Grade=0 when
# Grade is NaN, i.e. the taster recorded nothing in the Grade field.
mask_zero_comment = (
    panel_all["Comment"].astype(str).str.strip().isin(["0", "0.0"]) &
    panel_all["Grade"].isna()
)
n_zero = mask_zero_comment.sum()
panel_all.loc[mask_zero_comment, "Grade"]   = 0
panel_all.loc[mask_zero_comment, "Comment"] = pd.NA
print(f"Step 2 – Moved {n_zero:,} 'Comment=0' entries to Grade=0 (only where Grade was missing).")

# ── Step 3: Fix 'Absent' rows that have Grade > 0.5 ─────────────────────────
mask_absent_fix = (
    (panel_all["Grade_Description"] == "Absent") &
    (~panel_all["Grade"].isin([0, 0.5])) &
    (panel_all["Criteria"] != "Comment")
)
n_absent = mask_absent_fix.sum()
panel_all.loc[mask_absent_fix, "Grade"] = 0
print(f"Step 3 – Corrected {n_absent:,} 'Absent' rows with Grade > 0.5 → set to 0.")

# Ensure Grade is numeric after all edits
panel_all["Grade"] = pd.to_numeric(panel_all["Grade"], errors="coerce")

print(f"\nRows before grade validation: {n_rows_start:,}")
print(f"Rows after  grade validation: {len(panel_all):,}")

 AQ_Total is consistent with the sum of its components for all tasters.

Step 1 – Removed 1 rows where Comment == 'x'.  Rows remaining: 3,653
Step 2 – Moved 2 'Comment=0' entries to Grade=0 (only where Grade was missing).
Step 3 – Corrected 2 'Absent' rows with Grade > 0.5 → set to 0.

Rows before grade validation: 3,654
Rows after  grade validation: 3,653


---
## 9 · Fix Group Value `x` → `Savoury`

In [33]:
# One row carries Group = 'x' (data entry error).
# Based on the comment content ('Yeasty'), the correct group is 'Savoury'.

mask_x_group = panel_all["Group"] == "x"
n_changed    = mask_x_group.sum()

if n_changed > 0:
    print(f"Rows with Group == 'x' (before fix):")
    display(panel_all.loc[mask_x_group, ["Trial", "Section", "Taster", "Group", "Comment"]])
    panel_all.loc[mask_x_group, "Group"] = "Savoury"
    print(f"\n→ Changed {n_changed:,} row(s) from Group='x' to 'Savoury'.")
else:
    print("No rows with Group == 'x' found.")

Rows with Group == 'x' (before fix):


,Trial,Section,Taster,Group,Comment
2600,110019,Palate_Aromas_Other,CT,x,Yeasty



→ Changed 1 row(s) from Group='x' to 'Savoury'.


---
## 10 · Detect & Resolve Duplicate Grades

Every taster should appear **exactly once** per Trial-Section combination (for non-group-level sections). Two known data entry issues are resolved:

- **Trial 110001, Taster BP, `Nose_Aroma_Intensity`**: BP graded twice; PM's grade is missing. We remove BP's duplicate so neither taster is over-represented. Note that this means one taster's observation is missing for this Trial–Section.
- **Trial 110009, Taster LH, `Bitter`**: LH graded twice (Status 1 and Status 2). We keep only Status 2 (the revised, final grade).

In [34]:
# ── Sections where multiple rows per taster are valid (group-level sections) ─
# For these sections, the same taster can have rows for different groups → OK.
# These are the sections that use an open lexicon (tasters vote per word/group).
MULTI_ROW_SECTIONS = {
    "Mouthfeel_Chewed", "Mouthfeel_Initial",
    "Nose_Aromas_Dairy", "Nose_Aromas_Other",
    "Palate_Aromas_Dairy", "Palate_Aromas_Other",
    "Trigeminal_Notes", "Paste_Colour",
    "Paste_Structure", "Visual_Comments"
}

# ── Find duplicates in sections where only 1 row per taster is expected ───────
df_check = panel_all.loc[~panel_all["Section"].isin(MULTI_ROW_SECTIONS)].copy()
dupe_counts = (
    df_check
    .groupby(["Trial", "Taster", "Section"])
    .size()
    .reset_index(name="n_rows")
)
problem_cases = dupe_counts[dupe_counts["n_rows"] > 1]
print(f"Duplicate (Trial, Taster, Section) combinations found: {len(problem_cases)}")
if not problem_cases.empty:
    display(problem_cases)
    problem_detail = df_check.merge(
        problem_cases[["Trial", "Taster", "Section"]],
        on=["Trial", "Taster", "Section"], how="inner"
    ).sort_values(["Trial", "Taster", "Section"])
    display(problem_detail)

# ── Fix 1: Remove BP's duplicate in Trial 110001 / Nose_Aroma_Intensity ───────
mask_bp_dupe = (
    (panel_all["Section_ID"] == "4") &
    (panel_all["Trial"]      == "110001") &
    (panel_all["Taster"]     == "BP")
)
n_removed = mask_bp_dupe.sum()
panel_all = panel_all.loc[~mask_bp_dupe].copy()
print(f"\nFix 1 – Removed {n_removed:,} duplicate row(s) for BP in Trial 110001 / Nose_Aroma_Intensity.")

# ── Fix 2: Keep only Status 2 (final grade) for LH in Trial 110009 / Bitter ──
mask_lh = (
    (panel_all["Section_ID"] == "15") &
    (panel_all["Trial"]      == "110009")
)
df_lh_sub  = panel_all.loc[mask_lh].copy()
df_lh_rest = panel_all.loc[~mask_lh].copy()

# Sort by Status ascending so Status=1 comes first, then keep last (= Status 2)
df_lh_sub = df_lh_sub.sort_values("Status")
df_lh_sub = df_lh_sub.drop_duplicates(
    subset=["Trial", "Section_ID", "Taster"], keep="last"
)
panel_all = pd.concat([df_lh_rest, df_lh_sub], ignore_index=True)
print(f"Fix 2 – Kept only Status=2 for LH in Trial 110009 / Bitter. "
      f"Rows now: {len(panel_all):,}")

# ── Fix 3: Remove rows where Criteria != 'Comment' but Grade is missing ───────
# These are scored rows with no grade value — they cannot be used for analysis.
# Comment rows are excluded from this check since they never carry a grade.
mask_missing_grade = (
    (panel_all["Criteria"] != "Comment") &
    panel_all["Grade"].isna()
)
n_missing = mask_missing_grade.sum()
print(f"\nFix 3 – Rows with missing Grade outside Comment sections: {n_missing:,}")
if n_missing > 0:
    display(panel_all.loc[mask_missing_grade, ["Trial", "Section", "Taster", "Criteria", "Grade"]])
panel_all = panel_all.loc[~mask_missing_grade].copy()
print(f"Removed {n_missing:,} rows. Rows remaining: {len(panel_all):,}")

Duplicate (Trial, Taster, Section) combinations found: 2


,Trial,Taster,Section,n_rows
15,110001,BP,Nose_Aroma_Intensity,2
302,110009,LH,Bitter,2


,Header_ID,Header,Section_ID,Section,Criteria,Trial,Taster,Status,Group,Comment,Grade,Grade_Description
2,2,Nose,4,Nose_Aroma_Intensity,Obs,110001,BP,1,<NA>,Nose intensity,2.0,Mild
3,2,Nose,4,Nose_Aroma_Intensity,Obs,110001,BP,1,<NA>,Nose intensity,3.0,Moderate
0,3,Tastes,15,Bitter,Obs,110009,LH,2,<NA>,Grade,0.0,Absent
1,3,Tastes,15,Bitter,Obs,110009,LH,1,<NA>,Grade,1.0,Negligible/Partial



Fix 1 – Removed 2 duplicate row(s) for BP in Trial 110001 / Nose_Aroma_Intensity.
Fix 2 – Kept only Status=2 for LH in Trial 110009 / Bitter. Rows now: 3,650

Fix 3 – Rows with missing Grade outside Comment sections: 1


,Trial,Section,Taster,Criteria,Grade
3494,110023,Context_Quality,DB,AQ,NaN


Removed 1 rows. Rows remaining: 3,649


---
## 11 · Normalise Group Names

Group values are free-text entries from the tasters. We convert them to `Title_Case_With_Underscores` to ensure consistent spelling and easy merging later.

In [35]:
def normalise_group(value):
    """Convert raw group label to Title_Case_With_Underscores."""
    if pd.isna(value):
        return pd.NA
    s = str(value).strip()
    s = s.lower()
    s = re.sub(r'[\/,]+', '_', s)   # slash / comma → underscore
    s = re.sub(r'\s+',    '_', s)   # whitespace   → underscore
    s = re.sub(r'_+',     '_', s)   # collapse multiple underscores
    s = s.strip('_')
    s = s.title()                   # Title Case per word segment
    return s

panel_all["Group"] = panel_all["Group"].apply(normalise_group).astype("string")

# Replace the string literal '<Na>' that pandas sometimes produces → actual NA
panel_all["Group"] = panel_all["Group"].replace("<Na>", pd.NA)

print("Top Group values after normalisation:")
display(panel_all["Group"].value_counts().head(15))

Top Group values after normalisation:


Group
Sweet_Dairy        144
Butter_Flavours    136
Firm               103
Sour_Dairy         101
Fruit               84
Smooth              77
Irritant            73
Granular            67
Bakery              64
Meat                64
Vegetal             59
Elastic             56
Spice               54
Cheese              52
Woodland_Field      50
Name: count, dtype: Int64

---
## 12 · Fix Incorrect `Header_ID` and `Section_ID` Values

In [36]:
# ── Header_ID corrections ─────────────────────────────────────────────────────
# 'Palate' was incorrectly stored with Header_ID = 3 (should be 4)
mask_palate = (panel_all["Header"] == "Palate") & (panel_all["Header_ID"] == "3")
n_palate    = mask_palate.sum()
panel_all.loc[mask_palate, "Header_ID"] = "4"

# 'Nose' was incorrectly stored with Header_ID = 1 (should be 2)
mask_nose   = (panel_all["Header"] == "Nose") & (panel_all["Header_ID"] == "1")
n_nose      = mask_nose.sum()
panel_all.loc[mask_nose, "Header_ID"] = "2"

print(f"Header_ID corrected: {n_palate:,} Palate rows (3→4), {n_nose:,} Nose rows (1→2).")

# ── Section_ID corrections ────────────────────────────────────────────────────
# Palate_Aromas_Dairy and Palate_Aromas_Other have no IDs assigned since these were newly added 
panel_all.loc[panel_all["Section"] == "Palate_Aromas_Dairy", "Section_ID"] = "35"
panel_all.loc[panel_all["Section"] == "Palate_Aromas_Other", "Section_ID"] = "36"

print("Section_IDs 35/36 assigned to Palate_Aromas_Dairy / Palate_Aromas_Other.")

# Verification: each Section should map to exactly one Section_ID
sid_check = panel_all.groupby("Section")["Section_ID"].nunique()
if (sid_check > 1).any():
    print("\n  Sections with more than one Section_ID:")
    display(sid_check[sid_check > 1])
else:
    print("\n All sections have exactly one Section_ID.")

print("\nSection → ID mapping:")
display(
    panel_all[["Section", "Section_ID"]]
    .drop_duplicates()
    .sort_values("Section_ID")
)

Header_ID corrected: 1 Palate rows (3→4), 60 Nose rows (1→2).
Section_IDs 35/36 assigned to Palate_Aromas_Dairy / Palate_Aromas_Other.

 All sections have exactly one Section_ID.

Section → ID mapping:


,Section,Section_ID
0,Visual_Comments,1
32,Palate_Intensity,10
17,Savoury_Umami,11
18,Salt,12
19,Sweet,13
20,Acid_Sour,14
415,Bitter,15
21,Trigeminal_Intensity,16
33,Trigeminal_Notes,17
38,Mouthfeel_Initial,18


---
## 13 · Validate Taster Completeness

Every Trial-Section combination should ideally have grades from all 5 tasters. This cell reports any combinations that are missing tasters (no rows are removed here).

In [37]:
completeness = (
    panel_all
    .groupby(["Trial", "Section"])["Taster"]
    .apply(lambda x: set(x.dropna().unique()))
    .reset_index(name="Present_Tasters")
)

completeness["Missing_Tasters"] = completeness["Present_Tasters"].apply(
    lambda s: EXPECTED_TASTERS - s
)
completeness["n_present"] = completeness["Present_Tasters"].apply(len)

incomplete = completeness[completeness["n_present"] < len(EXPECTED_TASTERS)]
pct = len(incomplete) / len(completeness) * 100

print(f"Total Trial–Section combinations : {len(completeness):,}")
print(f"Combinations with < 5 tasters   : {len(incomplete):,} ({pct:.1f}%)")

if not incomplete.empty:
    print("\nDetail:")
    display(
        incomplete[["Trial", "Section", "n_present", "Missing_Tasters"]]
        .sort_values(["Trial", "Section"])
    )

Total Trial–Section combinations : 396
Combinations with < 5 tasters   : 20 (5.1%)

Detail:


,Trial,Section,n_present,Missing_Tasters
17,110001,Nose_Aroma_Intensity,3,"{BP, PM}"
18,110001,Nose_Aromas_Dairy,4,{PM}
19,110001,Nose_Aromas_Other,4,{PM}
23,110001,Paste_Colour,4,{PM}
24,110001,Paste_Structure,4,{PM}
32,110001,Visual_Comments,4,{PM}
82,110009,Mouthfeel_Initial,4,{LH}
148,110020,Mouthfeel_Initial,4,{LH}
163,110020,Trigeminal_Notes,4,{BP}
181,110021,Mouthfeel_Initial,4,{PM}


---
## 14 · Save Output

`panel_all` is saved as `panel_cleaned.csv`. Notebook 2 loads this file directly — no need to re-run the cleaning pipeline.

In [38]:
panel_all.to_csv(OUT_PANEL_CLEANED, index=False)
print(f" Saved: {OUT_PANEL_CLEANED}  "
      f"({panel_all.shape[0]:,} rows × {panel_all.shape[1]} columns)")
print("\nReady for Notebook 2.")

 Saved: panel_cleaned.csv  (3,649 rows × 12 columns)

Ready for Notebook 2.


---
## 15 · Final Quality Report

Sanity check on `panel_all` (= `panel_cleaned.csv`) before handing it to Notebook 2.

In [39]:
display(Markdown("### `panel_cleaned` – Cleaned Taster-Level Panel Data"))

panel_all.info()

print("\nDescriptive statistics:")
display(panel_all.describe(include="all").T)

print(f"\nTrials   : {sorted(panel_all['Trial'].unique())}")
print(f"Sections : {sorted(panel_all['Section'].unique())}")
print(f"Tasters  : {sorted(panel_all['Taster'].dropna().unique())}")
print(f"Statuses : {sorted(panel_all['Status'].dropna().unique())}")

print("\nMissing values per column:")
display(panel_all.isna().sum().sort_values(ascending=False))

print("\nOccurrences per Trial × Section:")
display(
    pd.crosstab(panel_all["Trial"], panel_all["Section"],
                margins=True, margins_name="Sum")
)


### `panel_cleaned` – Cleaned Taster-Level Panel Data

<class 'pandas.core.frame.DataFrame'>
Index: 3649 entries, 0 to 3649
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Header_ID          3649 non-null   string 
 1   Header             3649 non-null   object 
 2   Section_ID         3649 non-null   string 
 3   Section            3649 non-null   object 
 4   Criteria           3649 non-null   string 
 5   Trial              3649 non-null   string 
 6   Taster             3649 non-null   string 
 7   Status             3649 non-null   Int64  
 8   Group              2105 non-null   string 
 9   Comment            2632 non-null   string 
 10  Grade              3349 non-null   float64
 11  Grade_Description  3649 non-null   string 
dtypes: Int64(1), float64(1), object(2), string(8)
memory usage: 374.2+ KB

Descriptive statistics:


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Header_ID,3649,9,4,1251,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Header,3649,9,Palate,1251,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Section_ID,3649,33,36,432,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Section,3649,33,Palate_Aromas_Other,432,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Criteria,3649,5,Obs,2926,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Trial,3649,12,120003,345,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Taster,3649,5,CT,772,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Status,3649.0,<NA>,<NA>,<NA>,1.109071,0.321297,1.0,1.0,1.0,1.0,3.0
Group,2105,67,Sweet_Dairy,144,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Comment,2632,957,Pale yellow,55,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Trials   : ['110001', '110002', '110009', '110019', '110020', '110021', '110022', '110023', '110024', '120003', '120004', '240988']
Sections : ['AQ_Total', 'Acid_Sour', 'Aftertastes', 'Balance', 'Bitter', 'Complexity', 'Condition', 'Context', 'Context_Evidence', 'Context_Evidence_Quality', 'Context_Quality', 'Degree_of_Artificiality', 'Development', 'Interest', 'Length', 'Mouthfeel_Chewed', 'Mouthfeel_Initial', 'Nose_Aroma_Intensity', 'Nose_Aromas_Dairy', 'Nose_Aromas_Other', 'Palate_Aromas_Dairy', 'Palate_Aromas_Other', 'Palate_Intensity', 'Paste_Colour', 'Paste_Structure', 'Processed_Taste', 'Salt', 'Satisfaction', 'Savoury_Umami', 'Sweet', 'Trigeminal_Intensity', 'Trigeminal_Notes', 'Visual_Comments']
Tasters  : ['BP', 'CT', 'DB', 'LH', 'PM']
Statuses : [np.int64(1), np.int64(2), np.int64(3)]

Missing values per column:


Group                1544
Comment              1017
Grade                 300
Header_ID               0
Section                 0
Section_ID              0
Header                  0
Criteria                0
Status                  0
Taster                  0
Trial                   0
Grade_Description       0
dtype: int64


Occurrences per Trial × Section:


Section,AQ_Total,Acid_Sour,Aftertastes,Balance,Bitter,Complexity,Condition,Context,Context_Evidence,Context_Evidence_Quality,Context_Quality,Degree_of_Artificiality,Development,Interest,Length,Mouthfeel_Chewed,Mouthfeel_Initial,Nose_Aroma_Intensity,Nose_Aromas_Dairy,Nose_Aromas_Other,Palate_Aromas_Dairy,Palate_Aromas_Other,Palate_Intensity,Paste_Colour,Paste_Structure,Processed_Taste,Salt,Satisfaction,Savoury_Umami,Sweet,Trigeminal_Intensity,Trigeminal_Notes,Visual_Comments,Sum
Trial,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
110001,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,11,13,3,29,31,31,31,5,12,29,5,5,5,5,5,5,23,5,328
110002,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,10,7,5,25,23,24,31,5,10,24,5,5,5,5,5,5,12,5,286
110009,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,16,11,5,28,25,28,31,5,8,20,5,5,5,5,5,5,18,7,307
110019,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,13,13,5,25,18,25,47,5,8,27,5,5,5,5,5,5,16,5,312
110020,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,15,6,5,20,17,32,33,5,6,23,5,5,5,5,5,5,10,5,282
110021,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,9,8,5,16,11,39,35,5,9,17,5,5,5,5,5,5,12,5,276
110022,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,11,12,5,23,19,25,11,5,7,22,5,5,5,5,5,5,7,5,257
110023,5,5,5,5,5,5,5,5,5,5,4,5,5,5,5,15,8,5,23,30,31,54,5,8,29,5,5,5,5,5,5,20,5,337
110024,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,10,12,5,26,27,23,42,5,10,31,5,5,5,5,5,5,14,6,316
